<a href="https://colab.research.google.com/github/Pascingo/CTA-media-analysis/blob/Timon/Stage_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:


!pip install -r https://raw.githubusercontent.com/Pascingo/CTA-media-analysis/main/requirements.txt
!pip install gensim

import os
import glob
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import spacy
import json
import time
import textwrap
import re
import langextract as lx
from textblob import TextBlob
from sklearn.feature_extraction.text import CountVectorizer
from tqdm.auto import tqdm
from google.colab import userdata
import gdown
import networkx as nx
from pyvis.network import Network
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from google.colab import drive



activate_cache = True
actualize_cache = True

if activate_cache:

else:
  # Settings we use in several cells
  TEXT_COLUMN = "content"
  START_DATE = "2024-09-01"
  END_DATE = "2026-08-31"

  GOVERNANCE_KEYWORDS = [
      "security", "accountability", "bias", "fairness",
      "regulatory", "regulation", "governance", "policy",
      "ethics", "compliance", "law"
  ]
  # Minimum number of keyword hits for an article to count as "AI Governance"
  MIN_KEYWORD_HITS = 5


  # All files are stored in the folder "data" (in Colab and locally)
  DATA_FOLDER = "data"
  os.makedirs(DATA_FOLDER, exist_ok=True)

  # Public Google Drive folder of our team with the fixed dataset version and the spaCy cache
  DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/14cAxvtxuTfwENtSDCmGcRpl6qhl-z62z"

  # Fixed dataset version, so that all results can be reproduced
  DATASET_FILE = os.path.join(DATA_FOLDER, "ai_media_dataset_20260911.csv")
  SPACY_CACHE_FILE = os.path.join(DATA_FOLDER, "spacy_processed_text.csv")
  OUTPUT_FILE = os.path.join(DATA_FOLDER, "ai_governance_cleaned.csv")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 42.3 MB/s eta 0:00:00


In [2]:


if activate_cache:

else:
  #file_name = "ai_media_dataset_20260911.csv"
  df = pd.read_csv(DATASET_FILE)

  print(f"Original dataset shape: {df.shape}")
  print("Found columns:", df.columns.tolist())




  df = pd.read_csv(DATASET_FILE, usecols=["title", "date", "content", "domain", "url", "tags"])

  print(f"Original dataset shape: {df.shape}")
  df.head(3)




Original dataset shape: (47751, 7)
Found columns: ['Unnamed: 0', 'title', 'date', 'content', 'domain', 'url', 'tags']
Original dataset shape: (47751, 6)


,title,date,content,domain,url,tags
0,Top 10 AI Companies in UK & Europe,2024-09-11,['With the field of AI accelerating all the ti...,aimagazine,https://www.aimagazine.com/articles/top-10-ai-...,"['StabilityAI', 'GenerativeAI', 'ComputerVisio..."
1,“ Winner ” Takes Political Comedy Seriously,2024-09-11,"['This was in 2017, and she believed that the ...",newyorker,https://www.newyorker.com/culture/the-front-ro...,"['MilitaryAndDefense', 'Drone', 'Audio']"
2,Optimizing Sustainable Construction with Smart...,2024-09-11,['A new study is shedding light on the latest ...,azobuild,https://www.azobuild.com/news.aspx?newsID=23603,"['Finetuning', 'EnsembleLearning', 'SHAP', 'Pl..."


In [3]:


import html, re, unicodedata


if activate_cache:
  df = pd.read_parquet(f"{DATA_FOLDER}/cleaned.parquet")
else:
  # --- Basis-Bereinigung: für Word2Vec UND später das LLM ---
  def base_clean(t):
      t = html.unescape(t)                          # &amp; -> &
      t = re.sub(r"<[^>]+>", " ", t)                # HTML-Tags
      t = re.sub(r"https?://\S+|www\.\S+", " ", t)  # URLs
      t = re.sub(r"[\u2010-\u2015]", "-", t)        # alle Strich-Varianten -> "-"
      t = re.sub(r"[\u2018\u2019]", "'", t)         # typografische Apostrophe -> '
      t = re.sub(r"[\u201c\u201d]", '"', t)
      t = unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode()
      return re.sub(r"\s+", " ", t).strip()

  # --- Word2Vec-spezifisch ---
  def w2v_clean(t):
      t = t.lower()
      t = re.sub(r"'s\b", "", t)                                   # openai's -> openai
      t = re.sub(r"(?<=[a-z])-(?=[a-z])", " ", t)                  # london-based -> london based
      t = re.sub(r"(?<=[a-z0-9])-(?=[0-9])|(?<=[0-9])-(?=[a-z])", "_", t)  # gpt-4o -> gpt_4o
      t = t.replace("/", " ")
      t = re.sub(r"[^a-z0-9_\s.,!?]", " ", t)                      # restliche Symbole
      return re.sub(r"\s+", " ", t).strip()

  df["base_text"] = df[TEXT_COLUMN].apply(base_clean)
  df = df.drop_duplicates(subset=["base_text"])   # nach Bereinigung nochmals: fängt mehr Duplikate
  df["w2v_text"]  = df["base_text"].apply(w2v_clean)


  df.head(5)

  if actualize_cache:
    df.to_parquet(f"{DATA_FOLDER}/cleaned.parquet")



,title,date,content,domain,url,tags,base_text,w2v_text
0,Top 10 AI Companies in UK & Europe,2024-09-11,['With the field of AI accelerating all the ti...,aimagazine,https://www.aimagazine.com/articles/top-10-ai-...,"['StabilityAI', 'GenerativeAI', 'ComputerVisio...",['With the field of AI accelerating all the ti...,with the field of ai accelerating all the time...
1,“ Winner ” Takes Political Comedy Seriously,2024-09-11,"['This was in 2017, and she believed that the ...",newyorker,https://www.newyorker.com/culture/the-front-ro...,"['MilitaryAndDefense', 'Drone', 'Audio']","['This was in 2017, and she believed that the ...","this was in 2017, and she believed that the pu..."
2,Optimizing Sustainable Construction with Smart...,2024-09-11,['A new study is shedding light on the latest ...,azobuild,https://www.azobuild.com/news.aspx?newsID=23603,"['Finetuning', 'EnsembleLearning', 'SHAP', 'Pl...",['A new study is shedding light on the latest ...,a new study is shedding light on the latest in...
3,Celebrating student entrepreneurship at delta ...,2024-09-11,"['With this year’ s delta v Demo Day, the Mart...",news.mit,https://news.mit.edu/2024/celebrating-student-...,['Personalisation'],"['With this year' s delta v Demo Day, the Mart...","with this year s delta v demo day, the martin ..."
4,How GCCs are shaping the future of business,2024-09-11,['Enterprises are looking to Global capability...,mckinsey,https://www.mckinsey.com/capabilities/operatio...,"['GenerativeAI', 'DeepLearning', 'Metaverse']",['Enterprises are looking to Global capability...,enterprises are looking to global capability c...


Schritt 1: Splits (auf Artikelebene, geschichtet nach Monat)

In [4]:
from sklearn.model_selection import train_test_split

df["date"] = pd.to_datetime(df["date"], errors="coerce")
print(df["date"].isna().sum(), "Daten nicht lesbar")   # sollte 0 oder sehr klein sein
df = df.dropna(subset=["date"])
print(df["date"].dtype)   # datetime64[ns]


df["month"] = df["date"].dt.to_period("M").astype(str)

train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df["month"], random_state=42)
val_df, test_df   = train_test_split(temp_df, test_size=0.5, stratify=temp_df["month"], random_state=42)

df["split"] = "train"
df.loc[val_df.index, "split"] = "val"
df.loc[test_df.index, "split"] = "test"
df[["split"]].to_csv("splits.csv")   # einmal speichern, im ganzen Team dieselben Splits verwenden
print(df["split"].value_counts())

0 Daten nicht lesbar
datetime64[ns]
split
train    38148
test      4769
val       4768
Name: count, dtype: int64


Ohne eindeutige Artikel-ID-Spalte speichert ihr hier den Index. Sicherer ist es, die ID-Spalte mitzuspeichern, falls ihr eine habt.

**Schritt 2: Sätze und Tokens**

In [5]:
import re, nltk
nltk.download("stopwords")
from nltk.corpus import stopwords
STOP = set(stopwords.words("english"))

def to_sentences(text):
    sentences = []
    for s in re.split(r"(?<=[.!?])\s+", text):           # Satzgrenze = Punkt + Leerzeichen, "3.1" bleibt ganz
        toks = [t.strip(".") for t in re.findall(r"[a-z0-9_.]+", s)]
        toks = [t for t in toks
                if len(t) > 1
                and t not in STOP
                and not t.replace(".", "").isdigit()]   # reine Zahlen raus, "gpt_4o" bleibt
        if len(toks) >= 3:
            sentences.append(toks)
    return sentences

df["sentences"] = df["w2v_text"].apply(to_sentences)
train_sents = [s for doc in df.loc[df.split == "train", "sentences"] for s in doc]

n_tokens = sum(len(s) for s in train_sents)
print(f"{len(train_sents):,} Sätze, {n_tokens:,} Tokens im Train-Split")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


3,119,036 Sätze, 38,190,680 Tokens im Train-Split


Nehmt die NLTK-Stoppwortliste. Die Listen von gensim und sklearn enthalten Wörter wie «computer» und «system», die ihr in AI-Texten behalten wollt.

**Schritt 3: Mehrwortbegriffe erkennen**

In [6]:
from gensim.models.phrases import Phrases, Phraser

bigram = Phraser(Phrases(train_sents, min_count=10, threshold=10))
train_phr = [bigram[s] for s in train_sents]

# Kontrolle: welche Phrasen wurden gefunden?
phrases = sorted({t for s in train_phr for t in s if "_" in t})
print(len(phrases), phrases[:50])

77208 ['0.001_0.003', '0.001_indicating', '0.015_0.045', '0.01_0.03', '0.01_0.1', '0.0275_0.0639', '0.03_0.06', '0.04_0.27', '0.05_0.80', '0.07_0.28', '0.07_0.30', '0.0_0.25', '0.0_1.0', '0.1.3_7', '0.10_0.20', '0.10_0.50', '0.11_megapixel', '0.12_1.02', '0.13_1', '0.13_m', '0.19_0.91', '0.1_0.3', '0.1_0.5', '0.1_0.6', '0.1_1', '0.1_10', '0.1_24', '0.1_3', '0.1_3.5', '0.1_degree', '0.1_second', '0.1_the', '0.20_0.25', '0.21_0.92', '0.22_0.23', '0.22_0.32', '0.23_second', '0.25_0.5', '0.25_1.0', '0.25_degree', '0.29.2_2', '0.29_0.50', '0.2_0.3', '0.2_0.4', '0.2_0.5', '0.2_5', '0.2_inch', '0.30_0.35', '0.32_7.39', '0.330.72_0.33']


Wendet denselben bigram später auch auf Val und Test an. Er wird nur auf Train gelernt.

**Schritt 4: Training**

In [7]:

from gensim.models import Word2Vec

model = Word2Vec(
    sentences=train_phr,
    vector_size=200,
    window=5,
    min_count=5,
    sg=1,            # Skip-gram
    negative=10,
    epochs=15,
    workers=4,
    seed=42,
)
model.save("w2v_train.model")
print(f"Vokabular: {len(model.wv):,} Wörter")

Vokabular: 152,293 Wörter


In [18]:

from google.colab import drive
drive.mount("/content/drive")
!ls /content/drive/MyDrive

model.save("/content/drive/MyDrive/NLP_Projekt/w2v_train.model")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 CTA-media-analysis  'Nasviken - bjarne.gmap'   NLP_Projekt
 entities_spacy.csv   ner_processed_docs.csv


Mit workers=4 geht es schneller, die Ergebnisse schwanken aber minimal zwischen Läufen. Für volle Reproduzierbarkeit braucht es workers=1 und zusätzlich die Umgebungsvariable PYTHONHASHSEED. Für euer Projekt reicht es, das im Notebook kurz zu erwähnen.

In [8]:
key_terms = ["open_source", "llama", "regulation", "agent"]   # durch eure Track-Begriffe ersetzen
for term in key_terms:
    if term in model.wv:
        print(term, model.wv.get_vecattr(term, "count"), model.wv.most_similar(term, topn=5))
    else:
        print(term, "-> nicht im Vokabular")

open_source 13239 [('webanalyselosung_die', 0.7039892673492432), ('matomo_ist', 0.7013132572174072), ('datenhoheit_betont', 0.6789524555206299), ('devsecops_iteration', 0.6773271560668945), ('und_statistische', 0.6650152206420898)]
llama 1590 [('meta_llama', 0.6328611969947815), ('mistral_llama', 0.6291751265525818), ('qwen2.5_32b', 0.6230586171150208), ('llama3', 0.6126191020011902), ('mixtral', 0.6099010109901428)]
regulation 3811 [('regulations', 0.7077818512916565), ('legislation', 0.6837578415870667), ('regulatory_framework', 0.6638681888580322), ('regulation_eu', 0.6568753719329834), ('regulate', 0.6509333848953247)]
agent 45631 [('agents', 0.7806695103645325), ('agentic', 0.6619961857795715), ('coding_agent', 0.654756486415863), ('mcp_server', 0.6454384922981262), ('gateway_governs', 0.6121723055839539)]


So lest ihr das Ergebnis: Sinnvolle Nachbarn heissen, die Pipeline funktioniert. Bei Müll in den Nachbarn ist die Vorverarbeitung schuld. Fehlt ein wichtiger Begriff im Vokabular, kommt er zu selten vor oder wurde anders geschrieben, z. B. von Phrases mit einem anderen Wort zusammengefügt.

Schickt mir gern die Ausgabe von Schritt 2 (Anzahl Tokens) und vom Check, dann kann ich einschätzen, ob die Parameter passen.

In [20]:
import plotly.express as px
from sklearn.manifold import TSNE

seeds = ["regulation", "ai_act", "accountability", "bias", "fairness",
         "compliance", "open_source", "llama", "agent"]

rows = []
for s in seeds:
    if s not in model.wv:
        print(f"'{s}' nicht im Vokabular")
        continue
    rows.append((s, s, "Schlüsselbegriff"))
    for w, _ in model.wv.most_similar(s, topn=10):
        rows.append((w, s, "Nachbar"))

terms = pd.DataFrame(rows, columns=["word", "group", "kind"]).drop_duplicates("word")
vecs = np.array([model.wv[w] for w in terms["word"]])

coords = TSNE(n_components=2, perplexity=15, metric="cosine",
              init="pca", random_state=42).fit_transform(vecs)
terms["x"], terms["y"] = coords[:, 0], coords[:, 1]

fig = px.scatter(terms, x="x", y="y", color="group", symbol="kind",
                 text="word", hover_name="word", width=950, height=700,
                 title="Word2Vec: Schlüsselbegriffe und ihre nächsten Nachbarn (t-SNE)")
fig.update_traces(textposition="top center", textfont_size=9)
fig.show()

'ai_act' nicht im Vokabular


In [1]:
from sklearn.metrics import silhouette_score

TOPIC_COLUMN = "topic"   # <- Name eurer Stage-1-Topic-Spalte
N_SAMPLE = 3000          # Stichprobe, damit t-SNE schnell bleibt
TOP_TOPICS = 8           # nur die häufigsten Topics einfärben, Rest = "Other"

# 1. Dokumentvektor = Mittelwert der Wortvektoren eines Artikels
def doc_vector(sentences):
    toks = [t for s in sentences for t in bigram[s] if t in model.wv]
    if not toks:
        return None
    return np.mean([model.wv[t] for t in toks], axis=0)

sample = df.sample(n=min(N_SAMPLE, len(df)), random_state=42).copy()
sample["vec"] = sample["sentences"].apply(doc_vector)
sample = sample[sample["vec"].notna()].copy()
X = np.vstack(sample["vec"].values)

# 2. Labels: Stage-1-Topics, sonst KMeans als Ersatz
if TOPIC_COLUMN in sample.columns:
    labels = sample[TOPIC_COLUMN].astype(str)
    top = labels.value_counts().index[:TOP_TOPICS]
    sample["label"] = labels.where(labels.isin(top), "Other")
else:
    from sklearn.cluster import KMeans
    print(f"Spalte '{TOPIC_COLUMN}' nicht gefunden – verwende KMeans-Cluster")
    sample["label"] = KMeans(n_clusters=TOP_TOPICS, random_state=42, n_init=10).fit_predict(X).astype(str)

# 3. Wie gut trennen sich die Topics? (-1 = schlecht, 0 = starke Überlappung, 1 = klar getrennt)
sil = silhouette_score(X, sample["label"], metric="cosine")
print(f"Silhouette-Score: {sil:.3f}")

# 4. t-SNE auf 2D
coords = TSNE(n_components=2, perplexity=30, metric="cosine",
              init="pca", random_state=42).fit_transform(X)
sample["x"], sample["y"] = coords[:, 0], coords[:, 1]

# 5. Interaktive Karte (Hover zeigt Titel und Datum)
fig = px.scatter(sample, x="x", y="y", color="label",
                 hover_data={"title": True, "date": True, "x": False, "y": False},
                 opacity=0.6, width=950, height=700,
                 title=f"Word2Vec-Dokumentvektoren nach Topic (t-SNE, n={len(sample)}, Silhouette={sil:.2f})")
fig.update_traces(marker_size=4)
fig.show()

NameError: name 'df' is not defined